# Naïve DNA Compression — Guided Exercises
**AMLG | Information theory, FASTQ data, and introductory DNA compression**

This notebook is designed for a live teaching session **and independent practice**. We first explore how information content changes with nucleotide frequencies, then read sequencing data in FASTQ format and test a simple compression workflow.

### How to use this notebook
1. Run cells **top to bottom** with **Shift + Enter**.
2. For each ❓ exercise, read **Task**, **Hints**, **What this teaches**, and **Expected outcome**.
3. Fill in `# TODO` / `# YOUR CODE` sections yourself. The code is intentionally incomplete.
4. If you need help, open the commented lines under `# --- Solution (uncomment to use) ---` in the same code cell. **Do not run both your answer and the suggested solution.**
5. Compare your results with the expected outcome and try the optional checks.

**Learning route:** self-information → entropy → FASTQ records → FASTQ parsing → sequence compression → interpreting gzip results.

**Prerequisites:** basic Python (loops, strings, functions, lists, and dictionaries). Only the Python standard library is needed.

**Important:** This exercise uses a small demonstration FASTQ file if the original `data/example.fastq` is unavailable. Results for your own supplied FASTQ file may differ.

## Part 1 — Quantifying information
A repeated nucleotide string is more predictable than a string containing four equally frequent letters. We will measure this predictability with *entropy*.

# **Naïve Compression of DNA Sequences**

In [ ]:
from typing import Any, Dict, List

# Quantification of Information

Information theory is based on the observation that knowing that a likely event has occurred is less informative than knowing that an unlikely event has occurred.

A quantification of information should have the following properties:
- Likely events should have a low information content, and events that are certain to occur should have no information content at all. Less likely events should have a higher information content.
- Independent events should have additive information content.

The self-information of an event $x$ is hence defined as

$$I(x)=-\log{}P(x).$$

By using the base-2 logarithm, the unit of self-information is bit.
Hence, one bit is the amount of information gained by observing an event of probability $\frac{1}{2}$.

Self-information deals only with a single event $x$.
By computing the expectation of the self-information with respect to the entire probability distribution $P(\text{x})$ we obtain the entropy

$$H(\text{x})=\mathbb{E}_{\text{x}\sim{}P}[I(\text{x}=x)]=-\mathbb{E}_{\text{x}\sim{}P}[\log{}P(\text{x}=x)]=-\sum_{x}P(x)\log{}P(x).$$

The entropy gives the average information that is expected in an event $x$ drawn from probability distribution $P(\text{x})$.

### Intuition before the formula
- If the next nucleotide is **always A**, its identity is completely predictable: **0 bits per symbol** of uncertainty.
- If A and C are equally likely, identifying one requires **1 bit per symbol** on average.
- If A, C, G, and T are equally likely, the uncertainty is **2 bits per symbol**.

For a sequence with empirical frequencies $p(x)$, we compute Shannon entropy:

$$H(X)=-\sum_{x:p(x)>0}p(x)\log_2 p(x).$$

**Units:** bits **per nucleotide**, not total bits for the sequence. This estimate considers single-nucleotide frequencies only, not order or repeated patterns.

##### ❓ Computing entropy

Complete the function `entropy()` to compute the entropy of the given sequence.
Then, compute the entropy in bit per symbol of the sequences `AAAA`, `AACC`, `ACGT`.

**Task:** Implement `entropy(data, base=2)` using the counts for each symbol. Then report the entropy for `AAAA`, `AACC`, and `ACGT`.

**Hints:**
- Use `len(data)` for total symbols and `counts.values()` for symbol counts.
- Compute each $p=\mathrm{count}/\mathrm{total}$ and add `-p * math.log(p, base)`.
- `counts` never contains zero-frequency symbols, so no `log(0)` is needed.
- Base 2 means the answer is in **bits per symbol**.

**What this teaches:** How nucleotide-frequency distributions translate into uncertainty.

**Expected outcome:** `AAAA → 0.0`, `AACC → 1.0`, `ACGT → 2.0` bits per symbol.

In [ ]:
import collections
import math


def entropy(data: Any, base: int = 2) -> float:
    """Compute the entropy of a list of data."""
    if len(data) <= 1:
        return 0.0

    counts = collections.Counter()
    for datum in data:
        counts[datum] += 1

    eta = 0.0

    # TODO: iterate over the observed counts and accumulate -p * log_base(p).
    # YOUR CODE

    # --- Solution (uncomment to use) ---
    # for count in counts.values():
    #     p = count / len(data)
    #     eta -= p * math.log(p, base)

    return eta

In [ ]:
for sequence in ["AAAA", "AACC", "ACGT"]:
    # TODO: call entropy() and print the sequence and its entropy.
    # YOUR CODE
    pass  # Remove this placeholder when you write your answer.

    # --- Solution (uncomment to use) ---
    # print(f"{sequence}: {entropy(sequence):.1f} bits/symbol")

**Quick check (run after completing the exercises above).** The checks explain a mismatch rather than stopping the whole notebook.

In [ ]:
expected = {"AAAA": 0.0, "AACC": 1.0, "ACGT": 2.0}
for seq_test, target in expected.items():
    actual = entropy(seq_test)
    print(f"{seq_test}: got {actual:.3f}; expected {target:.3f} — "
          f"{'OK' if abs(actual-target) < 1e-9 else 'TRY AGAIN'}")

**Think about it:** `AAAA` and `AAAT` have different entropy. Why? Would `ACACACAC` and `AAAACCCC` have different *single-symbol* entropy if both have equal counts of A and C? (They do not: order is ignored by this entropy measure.)

## Part 2 — Understand the FASTQ format
Sequencing instruments typically produce short DNA fragments called **reads**. A FASTQ record stores both the nucleotide sequence and one quality character for each base.

# The FASTQ Format

The [FASTQ format](https://en.wikipedia.org/wiki/FASTQ_format) is the de-facto standard for storing both a biological sequence (usually nucleotide sequence) and its corresponding quality scores.
Both the sequence letter and quality score are each encoded with a single ASCII character.

Each sequence, i.e., read, is represented by a single FASTQ record, which consists of four lines:
- The first line contains the **read identifier**. It starts with `@`. Typically, sequencing machine vendors generate read identifiers in a proprietary systematic way.
- The second line contains the **sequence**, where each symbol is represented with a single ASCII character.
- The third line starts with `+` and contains an optional **description**. Usually this line is left empty; it then only contains `+` as separator between the sequence and the quality scores.
- The fourth line contains the **quality scores**. A quality score is a value indicating the confidence in a base call.

The following function can be used to convert a FASTQ record into a dictionary:

In [ ]:
def fastq_lines_to_dict(lines: List[str]) -> Dict[str, str]:
    """Convert a list of FASTQ lines to a dictionary."""
    keys = ["id", "seq", "desc", "qual"]
    return dict(zip(keys, lines))

### Worked example: one FASTQ record
```text
@read_01
ACGTAC
+
IIIIII
```
The four lines mean: (1) identifier, (2) sequence, (3) separator, and (4) encoded quality scores. `IIIIII` is a string of **quality characters**, not another DNA sequence. Each record normally has the same number of sequence symbols and quality symbols.

**Python reminder:** `line.rstrip("\n\r")` removes line endings. A list can collect four lines, then `fastq_lines_to_dict(lines)` gives a dictionary.

**Teaching simplification:** this parser expects simple, standard four-line records; production FASTQ handling may need additional validation and support for more formats.

### Example dataset setup

In [ ]:
from pathlib import Path

fastq_path = Path("data/example.fastq")

print("Using FASTQ file:", fastq_path)

##### ❓ Parsing a FASTQ file

Complete the following code to parse the FASTQ file `example.fastq`.
Print all FASTQ records in the following format:

```
Record 0: {'id': '@id0', 'seq': 'GATTTG...', 'desc': '+', 'qual': "!''*((..."}
Record 1: {'id': '@id1', 'seq': 'GATTTG...', 'desc': '+', 'qual': "!''*((..."}
...
```

**Task:** Complete `read_fastq_file()` so it accumulates four lines per record, converts them into a dictionary, and resets its temporary buffer.

**Hints:**
- After `lines.append(...)`, check `if len(lines) == 4:`.
- Append the result of `fastq_lines_to_dict(lines)` to `records`.
- Reset `lines = []` before processing the next FASTQ record.
- For real workflows, incomplete records should produce an error; see the optional validation check below.

**What this teaches:** How to parse a structured biological text file using a Python loop.

**Expected outcome:** With the generated teaching file, **four** dictionaries appear, with keys `id`, `seq`, `desc`, and `qual`. If you use the original course file, the count and values may differ.

In [ ]:
def read_fastq_file(file_path: str) -> List[Dict[str, str]]:
    """Read a FASTQ file and return a list of records."""
    with open(file=file_path, mode="r") as file:
        records = []
        lines = []
        for line in file:
            lines.append(line.rstrip())
            # TODO: when four lines have accumulated, append a record and clear lines.
            # YOUR CODE

            # --- Solution (uncomment to use) ---
            # if len(lines) == 4:
            #     records.append(fastq_lines_to_dict(lines))
            #     lines = []

        return records


records = read_fastq_file(file_path="data/example.fastq")
for i, record in enumerate(records):
    print(f"Record {i:2}: {record}")

**Check your parser:** A valid FASTQ record should have four fields, and sequence and quality strings should have equal length.

In [ ]:
if not records:
    print("No records parsed yet — complete the parser TODO above.")
else:
    print("Records parsed:", len(records))
    for i, rec in enumerate(records):
        ok = (set(rec) == {"id", "seq", "desc", "qual"}
              and rec["id"].startswith("@")
              and rec["desc"].startswith("+")
              and len(rec["seq"]) == len(rec["qual"]))
        print(f"Record {i}: {'OK' if ok else 'CHECK FORMAT'}")

## Part 3 — Compression of nucleotide sequences
**Idea:** First estimate the information content using nucleotide frequencies; then compress the same text with gzip and compare the number of bytes.

# Compression of Nucleotide Sequences

##### ❓ Compressing DNA sequence reads

Concatenate all nucleotide sequences from the FASTQ file `example.fastq`.
Compute the entropy (in bit per symbol) and the maximum (worst-case) compressed size in bit and byte.

> The assumption here is that every well-designed compressor that makes uses of any statistics beyond the per-symbol probabilites must yield a compressed bitstream that is smaller or equal to the entropy.

Then, use gzip to beat the estimated worst-case compression.

> Use the functions `gzip.compress()` and `gzip.decompress()`.
> Use UTF-8 encoding.

**Important interpretation note (clarifying the original exercise):** For an idealized long stream of **independent** symbols with a known distribution, the Shannon entropy is a theoretical *lower bound on average lossless coding length* for that model, **not a guaranteed maximum/worst-case gzip file size**. Real compressors also exploit repeated patterns, pay overhead for file headers, and operate on bytes. Therefore gzip **may be larger or smaller** than `sequence_length × single-symbol entropy`. The original wording above is retained for continuity, but we call this quantity the **entropy-based estimate**, rather than a guaranteed worst-case bound.

**Task:** (1) Join all `record["seq"]` strings into `seq`; (2) calculate entropy in bits per nucleotide; (3) calculate estimated bits and bytes; (4) encode the sequence as UTF-8; (5) compress/decompress with gzip; (6) verify you reconstructed the original sequence.

**Hints:**
- You can write `"".join(record["seq"] for record in records)`.
- Estimate bits as `len(seq) * entropy(seq)`; divide by **8** for bytes.
- `gzip.compress(seq.encode("utf-8"))` returns `bytes`.
- `gzip.decompress(...)` returns bytes; `.decode("utf-8")` returns text.
- `len(compressed)` gives the actual compressed **byte** count.

**What this teaches:** How theoretical compressibility differs from a practical lossless file compressor.

**Expected outcome:** You should see sequence length, entropy, entropy-based estimate, actual UTF-8 size, gzip size, and `round-trip correct: True`. Exact numerical values depend on the FASTQ file.

In [ ]:
import gzip

# Concatenate all sequences.
seq = ""
# YOUR CODE

# Compute the entropy (in bit per symbol) and the maximum (worst-case) compressed size
# in bit and byte.
# YOUR CODE

# Use gzip to beat the estimated worst-case compression.
# YOUR CODE

# TODO: print your calculations and confirm the compressed data round-trips.

# --- Solution (uncomment to use) ---
# seq = "".join(record["seq"] for record in records)
# h = entropy(seq)
# estimated_bits = len(seq) * h
# estimated_bytes = estimated_bits / 8
# raw_bytes = seq.encode("utf-8")
# compressed = gzip.compress(raw_bytes)
# restored = gzip.decompress(compressed).decode("utf-8")
# print("Nucleotides:", len(seq))
# print(f"Entropy: {h:.3f} bits/symbol")
# print(f"Entropy-based estimate: {estimated_bits:.2f} bits / {estimated_bytes:.2f} bytes")
# print("UTF-8 bytes:", len(raw_bytes))
# print("gzip bytes:", len(compressed))
# print("Round-trip correct:", restored == seq)


### Reflect: why might gzip not beat the estimate?
The word "compression" does not mean every input becomes smaller. Especially with short sequences, gzip's header and metadata create overhead. Also, nucleotide entropy based on **frequencies alone** does not capture order (e.g., long repetitive blocks). For longer or strongly patterned data, gzip might outperform a naive single-symbol entropy estimate because it exploits more structure.

**Optional challenge:** Compare `"A" * 1000` against `"ACGT" * 250`. Both are 1000 nucleotides, but the distributions differ. Then try `"AC" * 500` against `"A" * 500 + "C" * 500` — they have the same single-symbol entropy but different order. Does gzip produce equal file sizes? Why?

In [ ]:
# Optional exploration: intentionally complete (not assessed)
for label, sample in [
    ("all A", "A" * 1000),
    ("ACGT repeat", "ACGT" * 250),
    ("AC alternating", "AC" * 500),
    ("A block + C block", "A" * 500 + "C" * 500),
]:
    compressed_size = len(gzip.compress(sample.encode("utf-8"), mtime=0))
    print(f"{label:18} entropy={entropy(sample):.2f} bits/base; "
          f"gzip={compressed_size} bytes")


## Recap — What should you now be able to explain?
1. Why does `AAAA` have lower entropy than `ACGT`?
2. What do the four lines of a FASTQ record represent?
3. How can a loop collect multiple four-line FASTQ records?
4. Why do we encode a Python `str` into UTF-8 `bytes` before gzip compression?
5. Why is an entropy-based estimate **not** the same as actual gzip size?

**Extension (no coding required):** A FASTQ file stores quality strings and identifiers too. Our compression exercise compresses **only the concatenated nucleotide sequences**, not the complete FASTQ file. Would compression of the whole FASTQ have the same ratio?